# 08 Feature Engineering

## Objective

Objective:
Prepare model-ready features from the cleaned churn dataset using the findings from the earlier exploratory notebooks.

This step helps in:
- Converting cleaned data into modeling-friendly inputs
- Preserving strong churn signals discovered in bivariate and multivariate analysis
- Creating useful transformed and interaction features
- Preparing a consistent dataset for downstream feature selection and modeling


## Imports


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data

sns.set_theme(style="whitegrid")


## Load Clean Data

Load the cleaned churn dataset and separate the target, numerical, and categorical feature groups that will be used in feature engineering.


In [2]:
df = load_cleaned_data()

target_column = TARGET_COLUMN
numerical_columns = [column for column in NUMERICAL_COLUMNS if column in df.columns]
categorical_columns = [column for column in CATEGORICAL_COLUMNS if column in df.columns]

feature_groups = {
    "target_column": target_column,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "shape": df.shape,
}

feature_groups


{'target_column': 'Churn',
 'numerical_columns': ['tenure', 'MonthlyCharges', 'TotalCharges'],
 'categorical_columns': ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod'],
 'shape': (7043, 21)}

## Revisit EDA Findings

Before engineering new features, summarize the main findings from notebooks `05`, `06`, and `07` so the next steps stay tied to the actual churn patterns already observed.

- From `05_univariate_analysis.ipynb`, the dataset showed uneven category sizes, meaningful structural labels such as `No internet service`, and numerical distributions that may need careful transformation instead of naive scaling.
- From `06_bivariate_analysis.ipynb`, the strongest single-feature churn signals were `tenure`, `MonthlyCharges`, `TotalCharges`, `Contract`, `PaymentMethod`, `TechSupport`, `OnlineSecurity`, and `InternetService`, while weaker features such as `gender` and `PhoneService` showed much less separation.
- From `07_multivariate_analysis.ipynb`, `tenure` and `TotalCharges` were strongly related, and high-risk combinations such as `Month-to-month + Electronic check` and `Fiber optic + No TechSupport` suggested that interaction features may add value during modeling.

These findings guide the feature-engineering stage: preserve strong churn signals, handle redundant lifecycle features carefully, retain meaningful structural categories, and consider interaction features where the combined risk pattern is sharper than the single-feature pattern.


## Handle Categorical Encoding

Encode categorical features according to their business meaning.

- Binary features are mapped to `0/1`.
- `Contract` is ordinal-encoded because it reflects increasing customer commitment from `Month-to-month` to `Two year`.
- Multi-class service and payment features are one-hot encoded so structural labels such as `No internet service` remain distinct from ordinary `No` values.


In [3]:
encoded_df = df.copy()

binary_mappings = {
    "gender": {"Female": 0, "Male": 1},
    "Partner": {"No": 0, "Yes": 1},
    "Dependents": {"No": 0, "Yes": 1},
    "PhoneService": {"No": 0, "Yes": 1},
    "PaperlessBilling": {"No": 0, "Yes": 1},
    target_column: {"No": 0, "Yes": 1},
}

for column, mapping in binary_mappings.items():
    if column in encoded_df.columns:
        encoded_df[column] = encoded_df[column].map(mapping)

if "SeniorCitizen" in encoded_df.columns:
    encoded_df["SeniorCitizen"] = encoded_df["SeniorCitizen"].astype(int)

contract_mapping = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2,
}
if "Contract" in encoded_df.columns:
    encoded_df["Contract"] = encoded_df["Contract"].map(contract_mapping)

one_hot_columns = [
    column
    for column in [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "PaymentMethod",
    ]
    if column in encoded_df.columns
]

encoded_df = pd.get_dummies(
    encoded_df,
    columns=one_hot_columns,
    drop_first=False,
    dtype=int,
)

encoding_summary = {
    "binary_encoded": [column for column in binary_mappings if column in df.columns],
    "ordinal_encoded": [column for column in ["Contract"] if column in df.columns],
    "one_hot_encoded": one_hot_columns,
    "encoded_shape": encoded_df.shape,
}

encoding_summary


{'binary_encoded': ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'PaperlessBilling',
  'Churn'],
 'ordinal_encoded': ['Contract'],
 'one_hot_encoded': ['MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'PaymentMethod'],
 'encoded_shape': (7043, 40)}

In [4]:
encoded_df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,Contract,PaperlessBilling,MonthlyCharges,...,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0,0,1,0,1,0,0,1,29.85,...,1,0,0,1,0,0,0,0,1,0
1,5575-GNVDE,1,0,0,0,34,1,1,0,56.95,...,1,0,0,1,0,0,0,0,0,1
2,3668-QPYBK,1,0,0,0,2,1,0,1,53.85,...,1,0,0,1,0,0,0,0,0,1
3,7795-CFOCW,1,0,0,0,45,0,1,0,42.30,...,1,0,0,1,0,0,1,0,0,0
4,9237-HQITU,0,0,0,0,2,1,0,1,70.70,...,1,0,0,1,0,0,0,0,1,0
